In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [ ]:

df_train = pd.read_csv("Monsoon_Train.csv")
df_val   = pd.read_csv("Monsoon_Validation.csv")
df_test  = pd.read_csv("Monsoon_test.csv")
# final, diags = run_all(df_train, df_val, df_test)

In [ ]:

# 0. CONSTANTS
LAT, LON, TZ = 23.0031, 71.3923, 5.5
CAP = 21.5
SPD = 96                      # 15-min steps per day
HORIZON = 7 * SPD             # 672 steps: forecast is issued once, 7 days ahead
TARGET = "Act. Power (MW)"
OUT_DIR = "final_deliverables"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
def solar_geometry(ts, lat=LAT, lon=LON, tz=TZ):
    doy = ts.dt.dayofyear.values
    hour = (ts.dt.hour + ts.dt.minute / 60.0).values
    g = 2.0 * np.pi / 365.0 * (doy - 1 + (hour - 12.0) / 24.0)
    eqt = 229.18 * (0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g)
                    - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    decl = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g)
            - 0.006758*np.cos(2*g) + 0.000907*np.sin(2*g)
            - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))
    tst = hour*60.0 + eqt + 4.0*lon - 60.0*tz
    ha = np.deg2rad(tst/4.0 - 180.0)
    phi = np.deg2rad(lat)
    cosz = np.clip(np.sin(phi)*np.sin(decl) + np.cos(phi)*np.cos(decl)*np.cos(ha), -1, 1)
    elev = 90.0 - np.degrees(np.arccos(cosz))
    c = np.maximum(cosz, 0.01)
    ghi_cs = np.where(cosz > 0.01, 1098.0*c*np.exp(-0.057/c), 0.0)
    return np.maximum(cosz, 0.0), elev, ghi_cs


In [ ]:

def parse_ts(s):
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    formats = [
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M",
        "%d-%m-%Y %H:%M:%S", "%d-%m-%Y %H:%M",
        "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M",
        "%m/%d/%Y %H:%M:%S", "%m/%d/%Y %H:%M",
    ]
    for fmt in formats:
        m = out.isna()
        if not m.any():
            break
        out[m] = pd.to_datetime(s[m], format=fmt, errors="coerce")
    if out.isna().any():
        m = out.isna()
        out[m] = pd.to_datetime(s[m], dayfirst=True, format="mixed", errors="coerce")
    assert out.notna().all(), f"Unparsed timestamps remain: {s[out.isna()].unique()[:5]}"
    return out

In [ ]:
def build_features(df_train, df_val, df_test):
    parts = []
    for d, lab in [(df_train, "train"), (df_val, "validate"), (df_test, "test")]:
        d = d.copy()
        d["split"] = lab
        parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df["TimeStamp"] = parse_ts(df["TimeStamp"])
    df = df.sort_values("TimeStamp").drop_duplicates(subset=["TimeStamp"]).reset_index(drop=True)

    # Reindex only within contiguous blocks (breaks > 1 day are treated as separate seasonal blocks)
    block_id = (df["TimeStamp"].diff() > pd.Timedelta("1D")).cumsum()
    reindexed_blocks = []
    for _, blk in df.groupby(block_id):
        grid = pd.date_range(blk["TimeStamp"].min(), blk["TimeStamp"].max(), freq="15min")
        blk_re = blk.set_index("TimeStamp").reindex(grid).rename_axis("TimeStamp").reset_index()
        reindexed_blocks.append(blk_re)
    df = pd.concat(reindexed_blocks, ignore_index=True)

    # time encodings
    hr = (df["TimeStamp"].dt.hour + df["TimeStamp"].dt.minute / 60.0).values
    doy = df["TimeStamp"].dt.dayofyear.values
    df["sin_hour"], df["cos_hour"] = np.sin(2 * np.pi * hr / 24), np.cos(2 * np.pi * hr / 24)
    df["sin_doy"],  df["cos_doy"]  = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)

    # Compute lags safely (using skipna=True so minor 15-min gaps don't wipe out entire weeks)
    p = df[TARGET]
    df["lag_1d"] = p.shift(SPD)
    df["lag_7d"] = p.shift(HORIZON)
    df["slot_mean_7_13d"] = pd.concat(
        [p.shift(HORIZON + k * SPD) for k in range(7)], axis=1
    ).mean(axis=1, skipna=True)

    # Fallback if a seasonal split starts without 7-13 prior days in the same block
    df["lag_7d"] = df["lag_7d"].fillna(df["lag_1d"])
    df["slot_mean_7_13d"] = df["slot_mean_7_13d"].fillna(df["lag_7d"])

    # weather (S2+)
    rad = np.deg2rad(df["W-WDIR- (Degree)"].values)
    df["wind_u"] = -df["W-WSPD (m/s)"] * np.sin(rad)
    df["wind_v"] = -df["W-WSPD (m/s)"] * np.cos(rad)

    # physics / interaction (S3+)
    df["temp_diff"] = df["W-MODT- (°C)"] - df["W-AMBT (°C)"]
    df["ghi_temp_interaction"] = df["W-GHI (W/m2)"] * (1 - 0.004 * (df["W-MODT- (°C)"] - 25.0))
    df["wind_cooling_proxy"] = df["W-WSPD (m/s)"] * df["temp_diff"]

    # solar geometry (S4)
    cosz, elev, ghi_cs = solar_geometry(df["TimeStamp"])
    df["cos_zenith"], df["elevation_deg"], df["ghi_clearsky"] = cosz, elev, ghi_cs
    df["clear_sky_index"] = np.where(
        df["ghi_clearsky"] > 10,
        np.clip(df["W-GHI (W/m2)"] / np.maximum(df["ghi_clearsky"], 10), 0, 2),
        0.0,
    )

    # plant yield constant: fitted on TRAIN rows only
    m = (df["split"] == "train") & (df["W-GHI (W/m2)"] > 50)
    x = df.loc[m, "ghi_temp_interaction"]
    k_plant = float((df.loc[m, TARGET] * x).sum() / (x * x).sum())
    df["pv_physical_MW"] = np.clip(k_plant * df["ghi_temp_interaction"], 0, CAP)
    df["clear_sky_MW"]   = np.clip(k_plant * df["ghi_clearsky"], 0, CAP)
    return df

In [ ]:
TIME  = ["sin_hour", "cos_hour", "sin_doy", "cos_doy"]
AR    = ["lag_1d", "lag_7d", "slot_mean_7_13d"]
WX    = ["W-MODT- (°C)", "W-GHI (W/m2)", "W-AMBT (°C)", "W-WSPD (m/s)", "wind_u", "wind_v"]
PHYS  = ["temp_diff", "ghi_temp_interaction", "wind_cooling_proxy"]
SOLAR = ["cos_zenith", "elevation_deg", "ghi_clearsky", "clear_sky_index",
         "pv_physical_MW", "clear_sky_MW"]

FEATURE_SETS = {
    "S1": TIME + AR,
    "S2": TIME + AR + WX,
    "S3": TIME + AR + WX + PHYS,
    "S4": TIME + AR + WX + PHYS + SOLAR,
}
ALL_FEATS = FEATURE_SETS["S4"]


def split_data(df):
    ok = df.dropna(subset=ALL_FEATS + [TARGET])
    tr = ok[ok.split == "train"].reset_index(drop=True)
    va = ok[ok.split == "validate"].reset_index(drop=True)
    te = ok[ok.split == "test"].reset_index(drop=True)
    return tr, va, te

In [ ]:
def train_revisions(tr, va, te, feats):
    preds, diag = {}, []
    for rev, prm in REVISIONS.items():
        m = xgb.XGBRegressor(n_estimators=1000, early_stopping_rounds=30,
                             tree_method="hist", random_state=SEED, n_jobs=4, **prm)
        t0 = time.perf_counter()
        m.fit(tr[feats], tr[TARGET], eval_set=[(va[feats], va[TARGET])], verbose=False)
        fit_s = time.perf_counter() - t0
        row = {"Revision": rev, "best_iter": m.best_iteration + 1, "fit_time_s": fit_s}
        for name, d in [("train", tr), ("val", va), ("test", te)]:
            t1 = time.perf_counter()
            raw = m.predict(d[feats])
            if name == "test":
                row["predict_time_s"] = time.perf_counter() - t1   # inference on the 672 test rows
            p = postprocess(raw, d["TimeStamp"])
            mt = metrics(d[TARGET].values, p)
            row[f"MAE_{name}"], row[f"R2_{name}"] = mt["MAE"], mt["R2"]
            if name == "test":
                preds[rev] = p
        row["gap_test_minus_train_MAE"] = row["MAE_test"] - row["MAE_train"]
        diag.append(row)
    return preds, pd.DataFrame(diag)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def greedy_forward_xgb(tr, va, pool, min_gain=0.0005, full_search=True, prune=True):
    t_start = time.perf_counter()
    selected, remaining = [], list(pool)
    best_overall = np.inf
    best_subset = []
    no_improve_steps = 0
    patience = len(pool) if full_search else 3

    def score(feats):
        col_sub = 1.0 if len(feats) <= 3 else 0.8
        m = xgb.XGBRegressor(
            n_estimators=180,
            early_stopping_rounds=20,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=col_sub,
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )
        m.fit(tr[feats], tr[TARGET], eval_set=[(va[feats], va[TARGET])], verbose=False)
        p = postprocess(m.predict(va[feats]), va["TimeStamp"])
        return float(np.sqrt(mean_squared_error(va[TARGET], p)))

    print(f"\n--- Greedy Forward Selection (XGBoost | pool={len(pool)}) ---")
    step = 0
    while remaining:
        step += 1
        trial = {c: score(selected + [c]) for c in remaining}
        c = min(trial, key=trial.get)
        cand_score = trial[c]
        rel_gain = (best_overall - cand_score) / best_overall if np.isfinite(best_overall) else np.inf

        selected.append(c)
        remaining.remove(c)

        if rel_gain >= min_gain:
            best_overall = cand_score
            best_subset = list(selected)
            no_improve_steps = 0
            g_txt = "first feature" if not np.isfinite(rel_gain) else f"gain {rel_gain*100:.3f}%"
            print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  ({g_txt}) *NEW BEST*")
        else:
            no_improve_steps += 1
            print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  (gain {rel_gain*100:.3f}%)")
            if no_improve_steps >= patience:
                break

    selected = list(best_subset)

    if prune and len(selected) > 2:
        improved = True
        while improved and len(selected) > 2:
            improved = False
            for r in list(selected):
                cand = [f for f in selected if f != r]
                s_drop = score(cand)
                if s_drop <= best_overall:
                    selected = cand
                    best_overall = s_drop
                    improved = True
                    print(f"  [Prune]   - {r:<24s} val RMSE {best_overall:.4f}")
                    break

    if not selected and pool:
        selected = [pool[0]]

    elapsed_s = time.perf_counter() - t_start
    print(f"Final Selected ({len(selected)}/{len(pool)}) in {elapsed_s:.2f}s: {selected}\n")
    return selected, elapsed_s


def greedy_forward_rf(tr, va, pool, min_gain=0.0005, full_search=True, prune=True):
    t_start = time.perf_counter()
    selected, remaining = [], list(pool)
    best_overall = np.inf
    best_subset = []
    no_improve_steps = 0
    patience = len(pool) if full_search else 3

    def score(feats):
        m = RandomForestRegressor(
            n_estimators=60,
            max_depth=10,
            max_samples=0.7,
            random_state=42,
            n_jobs=-1,
        )
        m.fit(tr[feats], tr[TARGET])
        p = postprocess(m.predict(va[feats]), va["TimeStamp"])
        return float(np.sqrt(mean_squared_error(va[TARGET], p)))

    print(f"\n--- Greedy Forward Selection (Random Forest | pool={len(pool)}) ---")
    step = 0
    while remaining:
        step += 1
        trial = {c: score(selected + [c]) for c in remaining}
        c = min(trial, key=trial.get)
        cand_score = trial[c]
        rel_gain = (best_overall - cand_score) / best_overall if np.isfinite(best_overall) else np.inf

        selected.append(c)
        remaining.remove(c)

        if rel_gain >= min_gain:
            best_overall = cand_score
            best_subset = list(selected)
            no_improve_steps = 0
            g_txt = "first feature" if not np.isfinite(rel_gain) else f"gain {rel_gain*100:.3f}%"
            print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  ({g_txt}) *NEW BEST*")
        else:
            no_improve_steps += 1
            print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  (gain {rel_gain*100:.3f}%)")
            if no_improve_steps >= patience:
                break

    selected = list(best_subset)

    if prune and len(selected) > 2:
        improved = True
        while improved and len(selected) > 2:
            improved = False
            for r in list(selected):
                cand = [f for f in selected if f != r]
                s_drop = score(cand)
                if s_drop <= best_overall:
                    selected = cand
                    best_overall = s_drop
                    improved = True
                    print(f"  [Prune]   - {r:<24s} val RMSE {best_overall:.4f}")
                    break

    if not selected and pool:
        selected = [pool[0]]

    elapsed_s = time.perf_counter() - t_start
    print(f"Final Selected ({len(selected)}/{len(pool)}) in {elapsed_s:.2f}s: {selected}\n")
    return selected, elapsed_s

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# def greedy_forward(tr, va, pool, min_gain=0.0005, full_search=True, prune=True):
#     selected = []
#     remaining = list(pool)
#     best_overall = np.inf
#     best_subset = []
#     no_improve_steps = 0
#     patience = len(pool) if full_search else 3

#     def score(feats):
#         m = RandomForestRegressor(
#             n_estimators=60,
#             max_depth=10,
#             max_samples=0.7,
#             random_state=42,
#             n_jobs=-1,
#         )
#         m.fit(tr[feats], tr[TARGET])
#         p = postprocess(m.predict(va[feats]), va["TimeStamp"])
#         return float(np.sqrt(mean_squared_error(va[TARGET], p)))

#     print(f"\n--- Greedy Forward Selection (Random Forest | pool={len(pool)}, full_search={full_search}) ---")
#     step = 0
#     while remaining:
#         step += 1
#         trial = {c: score(selected + [c]) for c in remaining}
#         c = min(trial, key=trial.get)
#         cand_score = trial[c]
#         rel_gain = (best_overall - cand_score) / best_overall if np.isfinite(best_overall) else np.inf

#         selected.append(c)
#         remaining.remove(c)

#         if rel_gain >= min_gain:
#             best_overall = cand_score
#             best_subset = list(selected)
#             no_improve_steps = 0
#             g_txt = "first feature" if not np.isfinite(rel_gain) else f"gain {rel_gain*100:.3f}%"
#             print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  ({g_txt}) *NEW BEST*")
#         else:
#             no_improve_steps += 1
#             print(f"  [Step {step:02d}] + {c:<24s} val RMSE {cand_score:.4f}  (gain {rel_gain*100:.3f}%)")
#             if no_improve_steps >= patience:
#                 break

#     # Roll back to the exact subset that achieved the global lowest validation RMSE
#     selected = list(best_subset)

#     # Optional backward pruning: remove early features that became redundant once later ones entered
#     if prune and len(selected) > 2:
#         improved = True
#         while improved and len(selected) > 2:
#             improved = False
#             for r in list(selected):
#                 cand = [f for f in selected if f != r]
#                 s_drop = score(cand)
#                 if s_drop <= best_overall:
#                     selected = cand
#                     best_overall = s_drop
#                     improved = True
#                     print(f"  [Prune]   - {r:<24s} val RMSE {best_overall:.4f}  (dropped redundant feature)")
#                     break

#     if not selected and pool:
#         selected = [pool[0]]

#     print(f"Final Selected ({len(selected)}/{len(pool)}): {selected}\n")
#     return selected

In [ ]:
# =====================================================================
# 2. POST-PROCESSING, METRICS, DSM
# =====================================================================
def postprocess(pred, ts):
    p = np.clip(np.asarray(pred, float).ravel(), 0.0, CAP)
    hrs = (ts.dt.hour + ts.dt.minute/60.0).values
    p[(hrs >= 20.0) | (hrs <= 5.5)] = 0.0         # night rule
    return p


def gerc_dsm(actual, fc, cap=CAP):
    dev_mw = np.abs(actual - fc)
    pct = dev_mw / cap * 100.0
    kwh = dev_mw * 250.0
    e7, e15, e23 = (0.07*cap*250, 0.15*cap*250, 0.23*cap*250)
    ch = np.zeros_like(kwh)
    m1 = (pct > 7) & (pct <= 15);  ch[m1] = (kwh[m1] - e7) * 0.25
    m2 = (pct > 15) & (pct <= 23); ch[m2] = (e15-e7)*0.25 + (kwh[m2]-e15)*0.50
    m3 = pct > 23;                 ch[m3] = (e15-e7)*0.25 + (e23-e15)*0.50 + (kwh[m3]-e23)*0.75
    slab = np.select([pct <= 7, pct <= 15, pct <= 23],
                     ["<= 7%", "> 7% to <= 15%", "> 15% to <= 23%"], "> 23%")
    slab = np.where((actual == 0) & (fc == 0), "No Generation", slab)
    ch = np.where((actual == 0) & (fc == 0), 0.0, ch)
    return dev_mw, pct, kwh, slab, ch


def metrics(y, p):
    day = y > 0.05*CAP
    mape = np.mean(np.abs((y[day]-p[day])/y[day]))*100 if day.any() else np.nan
    active = (y > 0) | (p > 0)            # slots where something is generated/forecast
    dev_pct = np.abs(y-p)/CAP*100
    d=dict(
        MAE=mean_absolute_error(y, p),
        RMSE=float(np.sqrt(mean_squared_error(y, p))),
        MAPE=mape, R2=r2_score(y, p), MBE=float(np.mean(p-y)),
        DSM_7pct_all=float(np.mean(dev_pct <= 7)*100),
        DSM_7pct_daylight=float(np.mean(dev_pct[active] <= 7)*100) if active.any() else np.nan,
        MAE_daylight=mean_absolute_error(y[active], p[active]) if active.any() else np.nan)
    # print(d)
    return d

In [ ]:



# =====================================================================
# 5. DELIVERABLES (4 tables + overlay plot per scenario)
# =====================================================================
def export(tag, ts, y, preds, out_dir=OUT_DIR):
    date = ts.dt.strftime("%d-%b-%Y").values
    target_day = date[0]
    f1, f2, f3, f4 = [], [], [], []
    plt.figure(figsize=(15, 6)); plt.plot(ts, y, "k", lw=2, label="Actual")
    for rev, p in preds.items():
        dev_mw, pct, kwh, slab, ch = gerc_dsm(y, p)
        tt = ts.dt.strftime("%Y-%m-%d %H:%M")
        f1.append(pd.DataFrame({"Revision": rev, "Target_Time": tt,
                                "Actual_MW": y, "Forecast_MW": p}))
        f2.append(pd.DataFrame({"Revision": rev, "Target_Time": tt,
                                "Actual_MW": y, "Forecast_MW": p, "Error_MW": y-p,
                                "Deviation_Percent": pct, "Deviation_Energy_kWh": kwh,
                                "Violation_Slab": slab, "DSM_Charge_Rs": ch}))
        for d in np.unique(date):
            k = date == d
            f3.append({"Revision": rev, "Date": d, **metrics(y[k], p[k]),
                       "Total_DSM_Charge_Rs": ch[k].sum()})
        k = date == target_day
        f4.append({"Tenure": "Day",  "Scenario": tag, "Revision": rev,
                   "Date": target_day, **metrics(y[k], p[k])})
        f4.append({"Tenure": "Week", "Scenario": tag, "Revision": rev,
                   "Date": "Average", **metrics(y, p)})
        plt.plot(ts, p, "--", lw=1.1, label=rev)
    plt.title(f"{tag}: 7-day-ahead forecast (no test-period lags)"); plt.ylabel("MW")
    plt.grid(alpha=.3); plt.legend(loc="upper right"); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{tag}_overlay.png"), dpi=130)
    plt.show()
    pd.concat(f1).to_csv(os.path.join(out_dir, f"{tag}.csv"), index=False)
    pd.concat(f2).to_csv(os.path.join(out_dir, f"{tag}_DSM.csv"), index=False)
    pd.DataFrame(f3).to_csv(os.path.join(out_dir, f"{tag}_Metrics.csv"), index=False)
    return pd.DataFrame(f4)


In [ ]:
# def run_scenario_pipeline(
#     df_train,
#     df_validate,
#     df_test,
#     model_type="XGB",      # "XGB" or "RF"
#     scenario_name="S1",
#     use_greedy=False,
#     season="Monsoon",
#     target_day=None,
#     full_search=True,
# ):
#     tag = f"{scenario_name}_Greedy" if use_greedy else f"{scenario_name}_Standard"
#     approach_label = f"{model_type}_{tag}"
#     print(f"\n{'='*80}\nPreparing datasets for {approach_label} ({season})...\n{'='*80}")

#     t_prep0 = time.perf_counter()
#     df_all = build_features(df_train, df_validate, df_test)
#     train_proc, val_proc, test_proc = split_data(df_all)
#     prep_time_s = time.perf_counter() - t_prep0

#     pool = FEATURE_SETS[scenario_name]
#     greedy_time_s = 0.0
#     if use_greedy:
#         if model_type == "XGB":
#             features, greedy_time_s = greedy_forward_xgb(train_proc, val_proc, pool, min_gain=0.0005, full_search=full_search)
#         else:
#             features, greedy_time_s = greedy_forward_rf(train_proc, val_proc, pool, min_gain=0.0005, full_search=full_search)
#     else:
#         features = pool

#     X_train, y_train = train_proc[features], train_proc[TARGET].values
#     X_val, y_val = val_proc[features], val_proc[TARGET].values
#     X_test, y_test = test_proc[features], test_proc[TARGET].values

#     if model_type == "XGB":
#         rev_configs = [
#             ("Rev0", dict(n_estimators=350, max_depth=6, learning_rate=0.03,  subsample=0.80, colsample_bytree=0.8, random_state=42)),
#             ("Rev1", dict(n_estimators=350, max_depth=7, learning_rate=0.03,  subsample=0.80, colsample_bytree=0.8, random_state=101)),
#             ("Rev2", dict(n_estimators=400, max_depth=5, learning_rate=0.025, subsample=0.75, colsample_bytree=0.8, random_state=202)),
#             ("Rev3", dict(n_estimators=350, max_depth=6, learning_rate=0.035, subsample=0.80, colsample_bytree=0.7, random_state=303)),
#             ("Rev4", dict(n_estimators=450, max_depth=6, learning_rate=0.02,  subsample=0.85, colsample_bytree=0.8, random_state=404)),
#         ]
#     else:
#         rev_configs = [
#             ("Rev0", dict(n_estimators=150, max_depth=12, min_samples_split=4, random_state=42)),
#             ("Rev1", dict(n_estimators=200, max_depth=16, min_samples_split=3, random_state=101)),
#             ("Rev2", dict(n_estimators=250, max_depth=10, max_features="sqrt", min_samples_leaf=2, random_state=202)),
#             ("Rev3", dict(n_estimators=200, max_depth=14, max_samples=0.8, min_samples_split=4, random_state=303)),
#             ("Rev4", dict(n_estimators=300, max_depth=15, min_samples_split=2, random_state=404)),
#         ]

#     revisions_preds = {}
#     diag_rows = []
#     timing_rows = []

#     for rev_name, params in rev_configs:
#         print(f"Training {approach_label} - {rev_name}...")

#         # 1. TRAINING TIME
#         t0 = time.perf_counter()
#         if model_type == "XGB":
#             model = xgb.XGBRegressor(early_stopping_rounds=30, tree_method="hist", n_jobs=-1, **params)
#             model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
#             iters = model.best_iteration + 1
#         else:
#             model = RandomForestRegressor(n_jobs=-1, **params)
#             model.fit(X_train, y_train)
#             iters = params["n_estimators"]
#         train_s = time.perf_counter() - t0

#         # 2. TRAINING SET INFERENCE TIME
#         t_tr = time.perf_counter()
#         raw_train = model.predict(X_train)
#         p_train = postprocess(raw_train, train_proc["TimeStamp"])
#         train_infer_s = time.perf_counter() - t_tr

#         # 3. VALIDATION TIME (Inference + Evaluation)
#         t_va = time.perf_counter()
#         raw_val = model.predict(X_val)
#         p_val = postprocess(raw_val, val_proc["TimeStamp"])
#         val_s = time.perf_counter() - t_va

#         # 4. TESTING TIME (Inference + Evaluation)
#         t_te = time.perf_counter()
#         raw_test = model.predict(X_test)
#         p_test = postprocess(raw_test, test_proc["TimeStamp"])
#         test_s = time.perf_counter() - t_te

#         revisions_preds[rev_name] = p_test

#         mt_tr = metrics(y_train, p_train)
#         mt_va = metrics(y_val, p_val)
#         mt_te = metrics(y_test, p_test)

#         diag_rows.append({
#             "Revision": rev_name,
#             "trees_or_iter": iters,
#             "train_time_s": round(train_s, 4),
#             "val_time_s": round(val_s, 4),
#             "test_time_s": round(test_s, 4),
#             "MAE_train": round(mt_tr["MAE"], 4),
#             "R2_train": round(mt_tr["R2"], 4),
#             "MAE_val": round(mt_va["MAE"], 4),
#             "R2_val": round(mt_va["R2"], 4),
#             "MAE_test": round(mt_te["MAE"], 4),
#             "R2_test": round(mt_te["R2"], 4),
#             "gap_test_minus_train_MAE": round(mt_te["MAE"] - mt_tr["MAE"], 4),
#         })

#         timing_rows.append({
#             "Season": season,
#             "Model": model_type,
#             "Scenario": tag,
#             "Approach": approach_label,
#             "Revision": rev_name,
#             "n_features": len(features),
#             "trees_or_iter": iters,
#             "prep_time_s": round(prep_time_s, 4),
#             "greedy_search_s": round(greedy_time_s, 4),
#             "train_time_s": round(train_s, 4),
#             "train_infer_s": round(train_infer_s, 4),
#             "val_time_s": round(val_s, 4),
#             "test_time_s": round(test_s, 4),
#             "total_rev_time_s": round(train_s + val_s + test_s, 4),
#         })

#     print(f"\n=== {approach_label} ({len(features)}/{len(pool)} features) Diagnostics & Timing ===")
#     print(pd.DataFrame(diag_rows).to_string(index=False))

#     if target_day is None:
#         target_day = test_proc["TimeStamp"].dt.strftime("%d-%b-%Y").iloc[0]

#     file_tag = f"{season}_{approach_label}"
#     f4 = export(
#         tag=file_tag,
#         ts=test_proc["TimeStamp"],
#         y=y_test,
#         preds=revisions_preds,
#         out_dir=OUT_DIR,
#     )

#     return revisions_preds, f4, pd.DataFrame(timing_rows)


# # ==============================================================================
# # RUN ALL APPROACHES (XGBoost & Random Forest | Standard & Greedy | S1 - S4)
# # ==============================================================================
# SEASON_NAME = "Monsoon"   # Change to "Winter" when running Winter CSVs

# all_summaries = []
# all_timings = []

# for model_type in ["XGB", "RF"]:
#     for scen in ["S1", "S2", "S3", "S4"]:
#         # Standard approach
#         _, f4_std, tdf_std = run_scenario_pipeline(
#             df_train, df_val, df_test,
#             model_type=model_type,
#             scenario_name=scen,
#             use_greedy=False,
#             season=SEASON_NAME,
#         )
#         all_summaries.append(f4_std)
#         all_timings.append(tdf_std)

#         # Greedy approach
#         _, f4_grd, tdf_grd = run_scenario_pipeline(
#             df_train, df_val, df_test,
#             model_type=model_type,
#             scenario_name=scen,
#             use_greedy=True,
#             season=SEASON_NAME,
#             full_search=True,
#         )
#         all_summaries.append(f4_grd)
#         all_timings.append(tdf_grd)

# # Save master evaluation metrics
# master_summary = pd.concat(all_summaries, ignore_index=True)
# master_summary.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Summary.csv"), index=False)

# # Save detailed per-revision timing table
# timing_df = pd.concat(all_timings, ignore_index=True)
# timing_df.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Detailed_Timing.csv"), index=False)

# # Build and save aggregated per-approach timing comparison table
# approach_timing_summary = timing_df.groupby(["Model", "Scenario", "Approach"], sort=False).agg(
#     n_features=("n_features", "first"),
#     greedy_search_s=("greedy_search_s", "first"),
#     total_train_time_s=("train_time_s", "sum"),
#     mean_train_time_s=("train_time_s", "mean"),
#     total_val_time_s=("val_time_s", "sum"),
#     total_test_time_s=("test_time_s", "sum"),
#     mean_test_time_ms=("test_time_s", lambda s: s.mean() * 1000.0),
# ).round(4).reset_index()

# approach_timing_summary["grand_total_time_s"] = (
#     approach_timing_summary["greedy_search_s"]
#     + approach_timing_summary["total_train_time_s"]
#     + approach_timing_summary["total_val_time_s"]
#     + approach_timing_summary["total_test_time_s"]
# ).round(4)

# approach_timing_summary.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Timing_Summary.csv"), index=False)

# print("\n" + "=" * 115)
# print(f"{SEASON_NAME.upper()} COMPUTATION TIME SUMMARY ACROSS ALL APPROACHES (SECONDS / MS):")
# print("=" * 115)
# print(approach_timing_summary.to_string(index=False))

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# def run_scenario_pipeline(df_train, df_validate, df_test, scenario_name="S1", use_greedy=False, season="Monsoon", target_day=None):
#     tag = f"{scenario_name}_Greedy" if use_greedy else f"{scenario_name}_Standard"
#     print(f"Preparing contiguous datasets for {tag} ({season})...")
#     df_all = build_features(df_train, df_validate, df_test)
#     train_proc, val_proc, test_proc = split_data(df_all)

#     pool = FEATURE_SETS[scenario_name]
#     greedy_s = 0.0
#     if use_greedy:
#         t_g = time.perf_counter()
#         features = greedy_forward(train_proc, val_proc, pool, min_gain=0.002)
#         greedy_s = time.perf_counter() - t_g
#     else:
#         features = pool

#     X_train, y_train = train_proc[features].values, train_proc[TARGET].values
#     X_val, y_val = val_proc[features].values, val_proc[TARGET].values
#     X_test, y_test = test_proc[features].values, test_proc[TARGET].values

#     # Combine train + validation for final Random Forest fitting (matching your RF suite)
#     X_full = np.vstack([X_train, X_val])
#     y_full = np.concatenate([y_train, y_val])

#     revisions_preds = {}
#     diag_rows = []
#     timing_rows = []

#     if use_greedy:
#         timing_rows.append({
#             "Scenario": tag,
#             "Revision": "Greedy_Selection",
#             "n_features": len(features),
#             "n_trees": np.nan,
#             "fit_time_s": round(greedy_s, 4),
#             "predict_time_s": np.nan,
#         })

#     rev_configs = [
#         ("Rev0", dict(n_estimators=150, max_depth=12, min_samples_split=4, random_state=42)),
#         ("Rev1", dict(n_estimators=200, max_depth=16, min_samples_split=3, random_state=101)),
#         ("Rev2", dict(n_estimators=250, max_depth=10, max_features="sqrt", min_samples_leaf=2, random_state=202)),
#         ("Rev3", dict(n_estimators=200, max_depth=14, max_samples=0.8, min_samples_split=4, random_state=303)),
#         ("Rev4", dict(n_estimators=300, max_depth=15, min_samples_split=2, random_state=404)),
#     ]

#     for rev_name, params in rev_configs:
#         print(f"Training {rev_name}...")
#         t0 = time.perf_counter()
#         model = RandomForestRegressor(n_jobs=-1, **params)
#         model.fit(X_full, y_full)
#         fit_s = time.perf_counter() - t0

#         t1 = time.perf_counter()
#         raw_test = model.predict(X_test)
#         pred_s = time.perf_counter() - t1

#         p_train = postprocess(model.predict(X_train), train_proc["TimeStamp"])
#         p_val = postprocess(model.predict(X_val), val_proc["TimeStamp"])
#         p_test = postprocess(raw_test, test_proc["TimeStamp"])
#         revisions_preds[rev_name] = p_test

#         mt_tr = metrics(y_train, p_train)
#         mt_va = metrics(y_val, p_val)
#         mt_te = metrics(y_test, p_test)
#         diag_rows.append({
#             "Revision": rev_name,
#             "n_trees": params["n_estimators"],
#             "fit_time_s": round(fit_s, 4),
#             "MAE_train": round(mt_tr["MAE"], 4),
#             "R2_train": round(mt_tr["R2"], 4),
#             "MAE_val": round(mt_va["MAE"], 4),
#             "R2_val": round(mt_va["R2"], 4),
#             "predict_time_s": round(pred_s, 4),
#             "MAE_test": round(mt_te["MAE"], 4),
#             "R2_test": round(mt_te["R2"], 4),
#             "gap_test_minus_train_MAE": round(mt_te["MAE"] - mt_tr["MAE"], 4),
#         })
#         timing_rows.append({
#             "Scenario": tag,
#             "Revision": rev_name,
#             "n_features": len(features),
#             "n_trees": params["n_estimators"],
#             "fit_time_s": round(fit_s, 4),
#             "predict_time_s": round(pred_s, 4),
#         })

#     print(f"\n=== {tag} ({len(features)}/{len(pool)} features) overfit diagnostics ===")
#     print(pd.DataFrame(diag_rows).to_string(index=False))

#     if target_day is None:
#         target_day = test_proc["TimeStamp"].dt.strftime("%d-%b-%Y").iloc[0]

#     file_tag = f"{season}_RF_{tag}"
#     f4 = export(
#         tag=file_tag,
#         ts=test_proc["TimeStamp"],
#         y=y_test,
#         preds=revisions_preds,
#         out_dir=OUT_DIR,
#     )

#     print("\n" + "=" * 90)
#     print(f"{file_tag.upper()} EVALUATION TABLE (MATCHING REQUIRED SUBMISSION FORMAT):")
#     print("=" * 90)
#     print(f4.to_string(index=False))

#     return revisions_preds, f4, pd.DataFrame(timing_rows)


# # --- RUN ALL SCENARIOS (STANDARD & GREEDY) ---
# all_summaries = []
# all_timings = []

# for scen in ["S1", "S2", "S3", "S4"]:
#     t_s = time.perf_counter()
#     _, f4_std, tdf_std = run_scenario_pipeline(df_train, df_val, df_test, scenario_name=scen, use_greedy=False, season="Monsoon")
#     print(f"--> Total elapsed wall-clock time for {scen}_Standard: {time.perf_counter() - t_s:.2f}s\n")
#     all_summaries.append(f4_std)
#     all_timings.append(tdf_std)

#     t_g = time.perf_counter()
#     _, f4_grd, tdf_grd = run_scenario_pipeline(df_train, df_val, df_test, scenario_name=scen, use_greedy=True, season="Monsoon")
#     print(f"--> Total elapsed wall-clock time for {scen}_Greedy: {time.perf_counter() - t_g:.2f}s\n")
#     all_summaries.append(f4_grd)
#     all_timings.append(tdf_grd)

# master_summary = pd.concat(all_summaries, ignore_index=True)
# master_summary.to_csv(os.path.join(OUT_DIR, "Monsoon_RF_All_Scenarios_Summary.csv"), index=False)

# timing_df = pd.concat(all_timings, ignore_index=True)
# timing_df.to_csv(os.path.join(OUT_DIR, "Monsoon_RF_Timing_Table.csv"), index=False)

# print("\n" + "=" * 70)
# print("TOTAL TRAINING + GREEDY SEARCH TIME PER SCENARIO (SECONDS):")
# print("=" * 70)
# scen_totals = timing_df.groupby("Scenario", sort=False).agg(
#     n_features=("n_features", "first"),
#     total_fit_and_search_s=("fit_time_s", "sum"),
#     total_predict_s=("predict_time_s", "sum"),
# ).round(4)
# print(scen_totals.to_string())

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def run_scenario_pipeline(
    df_train,
    df_validate,
    df_test,
    model_type="XGB",      # "XGB" or "RF"
    scenario_name="S1",
    use_greedy=False,
    season="Monsoon",
    target_day=None,
    full_search=True,
):
    tag = f"{scenario_name}_Greedy" if use_greedy else f"{scenario_name}_Standard"
    approach_label = f"{model_type}_{tag}"
    file_tag = f"{season}_{approach_label}"

    # Nested subfolder structure: final_deliverables/xgb/s1_standard/ or final_deliverables/rf/s1_greedy/
    model_folder = model_type.lower()
    scen_folder = tag.lower()
    sub_out_dir = os.path.join(OUT_DIR, model_folder, scen_folder)
    os.makedirs(sub_out_dir, exist_ok=True)

    print(f"\n{'='*80}\nPreparing datasets for {file_tag} -> saving to '{sub_out_dir}/'...\n{'='*80}")

    t_prep0 = time.perf_counter()
    df_all = build_features(df_train, df_validate, df_test)
    train_proc, val_proc, test_proc = split_data(df_all)
    prep_time_s = time.perf_counter() - t_prep0

    pool = FEATURE_SETS[scenario_name]
    greedy_time_s = 0.0
    if use_greedy:
        if model_type == "XGB":
            features, greedy_time_s = greedy_forward_xgb(train_proc, val_proc, pool, min_gain=0.0005, full_search=full_search)
        else:
            features, greedy_time_s = greedy_forward_rf(train_proc, val_proc, pool, min_gain=0.0005, full_search=full_search)
    else:
        features = pool

    X_train, y_train = train_proc[features], train_proc[TARGET].values
    X_val, y_val = val_proc[features], val_proc[TARGET].values
    X_test, y_test = test_proc[features], test_proc[TARGET].values

    if model_type == "XGB":
        rev_configs = [
            ("Rev0", dict(n_estimators=350, max_depth=6, learning_rate=0.03,  subsample=0.80, colsample_bytree=0.8, random_state=42)),
            ("Rev1", dict(n_estimators=350, max_depth=7, learning_rate=0.03,  subsample=0.80, colsample_bytree=0.8, random_state=101)),
            ("Rev2", dict(n_estimators=400, max_depth=5, learning_rate=0.025, subsample=0.75, colsample_bytree=0.8, random_state=202)),
            ("Rev3", dict(n_estimators=350, max_depth=6, learning_rate=0.035, subsample=0.80, colsample_bytree=0.7, random_state=303)),
            ("Rev4", dict(n_estimators=450, max_depth=6, learning_rate=0.02,  subsample=0.85, colsample_bytree=0.8, random_state=404)),
        ]
    else:
        rev_configs = [
            ("Rev0", dict(n_estimators=150, max_depth=12, min_samples_split=4, random_state=42)),
            ("Rev1", dict(n_estimators=200, max_depth=16, min_samples_split=3, random_state=101)),
            ("Rev2", dict(n_estimators=250, max_depth=10, max_features="sqrt", min_samples_leaf=2, random_state=202)),
            ("Rev3", dict(n_estimators=200, max_depth=14, max_samples=0.8, min_samples_split=4, random_state=303)),
            ("Rev4", dict(n_estimators=300, max_depth=15, min_samples_split=2, random_state=404)),
        ]

    revisions_preds = {}
    diag_rows = []
    timing_rows = []

    for rev_name, params in rev_configs:
        print(f"Training {file_tag} - {rev_name}...")

        # 1. TRAINING TIME
        t0 = time.perf_counter()
        if model_type == "XGB":
            model = xgb.XGBRegressor(early_stopping_rounds=30, tree_method="hist", n_jobs=-1, **params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            iters = model.best_iteration + 1
        else:
            model = RandomForestRegressor(n_jobs=-1, **params)
            model.fit(X_train, y_train)
            iters = params["n_estimators"]
        train_s = time.perf_counter() - t0

        # 2. TRAINING SET INFERENCE TIME
        t_tr = time.perf_counter()
        raw_train = model.predict(X_train)
        p_train = postprocess(raw_train, train_proc["TimeStamp"])
        train_infer_s = time.perf_counter() - t_tr

        # 3. VALIDATION TIME (Inference + Evaluation)
        t_va = time.perf_counter()
        raw_val = model.predict(X_val)
        p_val = postprocess(raw_val, val_proc["TimeStamp"])
        val_s = time.perf_counter() - t_va

        # 4. TESTING TIME (Inference + Evaluation)
        t_te = time.perf_counter()
        raw_test = model.predict(X_test)
        p_test = postprocess(raw_test, test_proc["TimeStamp"])
        test_s = time.perf_counter() - t_te

        revisions_preds[rev_name] = p_test

        mt_tr = metrics(y_train, p_train)
        mt_va = metrics(y_val, p_val)
        mt_te = metrics(y_test, p_test)

        diag_rows.append({
            "Revision": rev_name,
            "trees_or_iter": iters,
            "train_time_s": round(train_s, 4),
            "val_time_s": round(val_s, 4),
            "test_time_s": round(test_s, 4),
            "MAE_train": round(mt_tr["MAE"], 4),
            "R2_train": round(mt_tr["R2"], 4),
            "MAE_val": round(mt_va["MAE"], 4),
            "R2_val": round(mt_va["R2"], 4),
            "MAE_test": round(mt_te["MAE"], 4),
            "R2_test": round(mt_te["R2"], 4),
            "gap_test_minus_train_MAE": round(mt_te["MAE"] - mt_tr["MAE"], 4),
        })

        timing_rows.append({
            "Season": season,
            "Model": model_type,
            "Scenario": tag,
            "Approach": approach_label,
            "Revision": rev_name,
            "n_features": len(features),
            "trees_or_iter": iters,
            "prep_time_s": round(prep_time_s, 4),
            "greedy_search_s": round(greedy_time_s, 4),
            "train_time_s": round(train_s, 4),
            "train_infer_s": round(train_infer_s, 4),
            "val_time_s": round(val_s, 4),
            "test_time_s": round(test_s, 4),
            "total_rev_time_s": round(train_s + val_s + test_s, 4),
        })

    diag_df = pd.DataFrame(diag_rows)
    timing_sub_df = pd.DataFrame(timing_rows)

    print(f"\n=== {file_tag} ({len(features)}/{len(pool)} features) Diagnostics & Timing ===")
    print(diag_df.to_string(index=False))

    # Save per-subpart diagnostics and timing CSVs inside sub_out_dir
    diag_df.to_csv(os.path.join(sub_out_dir, f"{file_tag}_Overfit_Diagnostics.csv"), index=False)
    timing_sub_df.to_csv(os.path.join(sub_out_dir, f"{file_tag}_Timing.csv"), index=False)

    if target_day is None:
        target_day = test_proc["TimeStamp"].dt.strftime("%d-%b-%Y").iloc[0]

    # Export all 4 GERC DSM deliverables and overlay plot directly into sub_out_dir
    f4 = export(
        tag=file_tag,
        ts=test_proc["TimeStamp"],
        y=y_test,
        preds=revisions_preds,
        out_dir=sub_out_dir,
    )
    f4.to_csv(os.path.join(sub_out_dir, f"{file_tag}_Tenure_Evaluation.csv"), index=False)

    return revisions_preds, f4, timing_sub_df


# ==============================================================================
# RUN ALL APPROACHES (XGBoost & Random Forest | Standard & Greedy | S1 - S4)
# ==============================================================================
SEASON_NAME = "Monsoon"   # Change to "Winter" when running Winter CSVs

all_summaries = []
all_timings = []

for model_type in ["XGB", "RF"]:
    for scen in ["S1", "S2", "S3", "S4"]:
        # Standard approach
        _, f4_std, tdf_std = run_scenario_pipeline(
            df_train, df_val, df_test,
            model_type=model_type,
            scenario_name=scen,
            use_greedy=False,
            season=SEASON_NAME,
        )
        all_summaries.append(f4_std)
        all_timings.append(tdf_std)

        # Greedy approach
        _, f4_grd, tdf_grd = run_scenario_pipeline(
            df_train, df_val, df_test,
            model_type=model_type,
            scenario_name=scen,
            use_greedy=True,
            season=SEASON_NAME,
            full_search=True,
        )
        all_summaries.append(f4_grd)
        all_timings.append(tdf_grd)

# Save master evaluation metrics in root OUT_DIR
master_summary = pd.concat(all_summaries, ignore_index=True)
master_summary.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Summary.csv"), index=False)

# Save master detailed per-revision timing table in root OUT_DIR
timing_df = pd.concat(all_timings, ignore_index=True)
timing_df.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Detailed_Timing.csv"), index=False)

# Build and save aggregated per-approach timing summary table in root OUT_DIR
approach_timing_summary = timing_df.groupby(["Model", "Scenario", "Approach"], sort=False).agg(
    n_features=("n_features", "first"),
    greedy_search_s=("greedy_search_s", "first"),
    total_train_time_s=("train_time_s", "sum"),
    mean_train_time_s=("train_time_s", "mean"),
    total_val_time_s=("val_time_s", "sum"),
    total_test_time_s=("test_time_s", "sum"),
    mean_test_time_ms=("test_time_s", lambda s: s.mean() * 1000.0),
).round(4).reset_index()

approach_timing_summary["grand_total_time_s"] = (
    approach_timing_summary["greedy_search_s"]
    + approach_timing_summary["total_train_time_s"]
    + approach_timing_summary["total_val_time_s"]
    + approach_timing_summary["total_test_time_s"]
).round(4)

approach_timing_summary.to_csv(os.path.join(OUT_DIR, f"{SEASON_NAME}_All_Approaches_Timing_Summary.csv"), index=False)

print("\n" + "=" * 115)
print(f"{SEASON_NAME.upper()} COMPUTATION TIME SUMMARY ACROSS ALL APPROACHES (SECONDS / MS):")
print("=" * 115)
print(approach_timing_summary.to_string(index=False))